# Web3 Graph Analysis in BigQuery

(a basic demo)

<table align="left">
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fcloudymoma%2Fgcp-playgroud-public%2Frefs%2Fheads%2Fmaster%2FBigQuery%crypto_graph.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Run in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/cloudymoma/gcp-playgroud-public/blob/master/BigQuery/crypto_graph.ipynb">
      <img width="32px" src="https://upload.wikimedia.org/wikipedia/commons/9/91/Octicons-mark-github.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

This workflow showcases a cloud-native approach to blockchain analytics by combining two powerful Google Cloud capabilities.

### The Power of BigQuery Graph

Historically, performing deep relationship analysis required extracting your tabular data and moving it into a specialized, third-party graph database. The newly introduced [BigQuery Graph](https://docs.cloud.google.com/bigquery/docs/graph-overview) eliminates this architectural headache by enabling zero-ETL graph analytics.

By applying a Property Graph schema (`CREATE PROPERTY GRAPH`) over your existing relational tables, you can use the ISO GQL standard to traverse millions of nodes directly in your data warehouse. In this notebook, we leveraged quantified path patterns (e.g., `->{1,3}`) and the native `%%bigquery --graph` notebook magic to visually trace multi-hop Ethereum transactions seamlessly, without ever leaving the BigQuery UI.

### The Advantage of BigQuery Public Datasets

Building this analysis from scratch would normally require running an Ethereum archive node, decoding smart contract bytecode, and maintaining complex, fragile data ingestion pipelines.

The [BigQuery Public Datasets](https://docs.cloud.google.com/bigquery/public-data) remove this friction entirely. Google hosts and continuously updates structured datasets for major blockchains (including Bitcoin, Ethereum, Solana, and Polygon). Because the data is pre-parsed and ready to query, you can focus purely on the business logic—like tracking stolen funds or mapping DeFi liquidity—rather than infrastructure. You only pay for the queries you run (with the first 1 TB per month free), making enterprise-grade Web3 intelligence accessible in minutes.



## Environment Setup & Dependencies 环境配置与依赖安装

This first block sets up our Colab Enterprise/Jupyter environment. We install the essential BigQuery Python client and the newly released `bigquery-magics` library (version 0.12+), which is critical because it contains the native `--graph` visualization engine for BigQuery. After installing the dependencies, we import the magics context and initialize our BigQuery client to communicate with Google Cloud.

In [ ]:
# Install a visualization library for interactive graphs
!pip install pandas db-dtypes bigquery-magics

from google.cloud import bigquery
from google.cloud.bigquery import magics
import pandas as pd

# Initialize the BigQuery client
client = bigquery.Client()

## Data Preparation: Nodes and Edges 数据准备：点与边

A Property Graph requires underlying relational tables to act as the raw materials. In this block, we write standard SQL to extract a subset of Ethereum data (from January 1st, 2026) out of the massive public dataset and store it in our personal GCP project.

The Edge Table (`eth_transfers`): Represents the "lines" connecting dots. It includes the sender, receiver, amount transferred (converted to ETH), and the gas costs.

The Node Table (`eth_addresses`): Represents the "dots" (wallets). We use `UNION DISTINCT` to grab every unique address that sent or received money on that day.

In [ ]:
project_id = 'du-hast-mich'
dataset_id = f'{project_id}.crypto_graph'

sql_prep = f"""
-- 1. Create the Edge Table (Transactions for one specific day to keep it small)
CREATE OR REPLACE TABLE `{dataset_id}.eth_transfers` AS
SELECT
    `hash` AS transaction_hash,
    from_address,
    to_address,
    value / POW(10, 18) AS eth_value,
    block_timestamp,
    receipt_gas_used,
    (gas_price / POW(10, 9)) AS gas_price_gwei -- Convert gas price to Gwei
FROM
    `bigquery-public-data.crypto_ethereum.transactions`
WHERE
    DATE(block_timestamp) = '2026-01-01'
    AND value > 0;

-- 2. Create the Vertex Table (Unique Addresses involved in those transactions)
CREATE OR REPLACE TABLE `{dataset_id}.eth_addresses` AS
SELECT DISTINCT address FROM (
    SELECT from_address AS address FROM `{dataset_id}.eth_transfers`
    UNION DISTINCT
    SELECT to_address AS address FROM `{dataset_id}.eth_transfers`
);
"""

# Execute the preparation query
job = client.query(sql_prep)
job.result()
print("Node and Edge tables created successfully!")

## Building the Property Graph Schema

This is where the magic happens. Using the new ISO Graph Query Language (GQL) standard, we execute a `CREATE PROPERTY GRAPH` DDL statement. This does not copy or move the data; instead, it lays a logical graph schema over our existing tables. We explicitly define the `eth_addresses` table as our `NODE TABLES` and the `eth_transfers` as our `EDGE TABLES`, mapping the source/destination relationships and exposing properties like `eth_value` and `gas_price_gwei` for analysis.

In [ ]:
sql_create_graph = f"""
CREATE OR REPLACE PROPERTY GRAPH `{dataset_id}.ethereum_graph`
  NODE TABLES (
    `{dataset_id}.eth_addresses` AS Wallet
      KEY (address)
      PROPERTIES (address)
  )
  EDGE TABLES (
    `{dataset_id}.eth_transfers` AS Transfer
      KEY (transaction_hash)
      SOURCE KEY (from_address) REFERENCES Wallet(address)
      DESTINATION KEY (to_address) REFERENCES Wallet(address)
      PROPERTIES (eth_value, block_timestamp, receipt_gas_used, gas_price_gwei)
  );
"""

job = client.query(sql_create_graph)
job.result()
print("Property Graph 'ethereum_graph' created successfully!")

## Visualizing a Fixed 2-Hop Relationship

Now we query the graph natively! We use the `%%bigquery --graph display_only` cell magic to trigger BigQuery's built-in visualization widget, while suppressing the raw text output.
Using the `MATCH` clause, we draw an "ASCII-art" style arrow pattern: `(a)-[t1]->(b)-[t2]->(c)`. This tells the engine to find exactly 2-hop money trails where funds moved from Wallet A to B, and then B to C, filtering for transfers strictly over 1.0 ETH. Returning the path as `TO_JSON(p)` allows the visualization canvas to render it interactively.

In [ ]:
%%bigquery --graph display_only

GRAPH `du-hast-mich.crypto_graph.ethereum_graph`

-- Assign the entire 2-hop money trail to the path variable 'p'
MATCH p = (a:Wallet)-[t1:Transfer]->(b:Wallet)-[t2:Transfer]->(c:Wallet)

-- Keep our filters for significant transactions
WHERE t1.eth_value > 1.0 AND t2.eth_value > 1.0

-- Return the entire path as a JSON object for the visualizer
RETURN TO_JSON(p) AS transaction_path

LIMIT 100

## Variable-Length Traversals (Quantified Path Patterns)

Finding rigid, fixed paths isn't always enough in Web3. In this final query, we use a Quantified Path Pattern by adding `{1, 3}` to our edge relationship. This tells the GQL engine to dynamically trace the flow of funds from a starting wallet to an ending wallet anywhere between 1 and 3 hops deep. Notice that we moved the `WHERE t.eth_value > 1.0` filter directly inside the edge brackets—this ensures that every single transfer in the variable-length chain meets our 1.0 ETH threshold.

In [ ]:
%%bigquery --graph display_only

GRAPH `du-hast-mich.crypto_graph.ethereum_graph`

-- The {1, 3} quantifier tells the engine to traverse 1 to 3 hops deep.
-- Notice that the WHERE filter is moved INSIDE the edge brackets so it evaluates every hop!
MATCH p = (start_node:Wallet)-[t:Transfer WHERE t.eth_value > 1.0]->{1, 3}(end_node:Wallet)

RETURN TO_JSON(p) AS transaction_path

LIMIT 100